In [2]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [54]:
import numpy as np

base_folder = f'/active-data/analysis_results/chr_pla/genus'
initial_label = 0.7

def temp_layer_link(previous_layer, next_layer, size_range = 10):
    temp_link1 = NMS_link[(NMS_link['source'].isin(previous_layer)) & (NMS_link['target'].isin(next_layer))]
    temp_link2 = NMS_link[(NMS_link['target'].isin(previous_layer)) & (NMS_link['source'].isin(next_layer))]
    temp_link2 = temp_link2.rename(columns={'source': 'target', 'target': 'source'})

    temp_link = pd.concat([temp_link1, temp_link2])
    temp_link = temp_link.drop_duplicates(subset=['source', 'target'], keep='first')
    temp_link = pd.merge(temp_link, replicon_data[['source', 'source_size']], on='source', how='left')
    temp_link = pd.merge(temp_link, replicon_data[['target', 'target_size']], on='target', how='left')
    temp_link['size_ratio'] = temp_link['source_size'] / temp_link['target_size']
    temp_link = temp_link[(temp_link['size_ratio'] < size_range) & (temp_link['size_ratio'] > 1/size_range)]
    return temp_link
        

for genus_name in keep_genus:
    try:
        layer_link = pd.read_csv(f'{base_folder}/figure_data/circle_layer_network_output/{genus_name}/all_dot_table.csv')
        NMS_link = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/{genus_name}_NMS_replicon_link.csv')
        replicon_data = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
        replicon_data['source'] = replicon_data['accession']
        replicon_data['source_size'] = replicon_data['size']
        replicon_data['target'] = replicon_data['accession']
        replicon_data['target_size'] = replicon_data['size']
        size_dict = dict(zip(replicon_data['accession'], replicon_data['size']))
    except:
        continue
    layer_link['layer_floor'] = layer_link['layer'].apply(np.floor)
    unique_layers = layer_link['layer_floor'].unique().tolist()
    unique_layers_sorted = sorted(unique_layers)

    zero_layer = layer_link[layer_link['layer'] == 0.7]['acc'].to_list()
    zero_link = temp_layer_link(zero_layer, layer_link[layer_link['layer_floor'] == 1]['acc'].to_list())
    zero_acc = zero_link['source'].unique().tolist()
    print(genus_name, max(unique_layers_sorted), zero_acc)
    for acc in zero_acc:
        for layer in unique_layers_sorted[1:]:
            if layer == 1:
                previous_layer = [acc]
            else:
                previous_layer = next_layer
            next_layer = layer_link[layer_link['layer_floor'] == layer]['acc'].to_list()
            temp_link = temp_layer_link(previous_layer, next_layer)
            next_layer = temp_link['target'].unique().tolist()
            if len(temp_link) == 0:
                print(acc, size_dict[acc], layer-1)
                break
            elif layer == max(unique_layers_sorted):
                print(acc, size_dict[acc], layer)

Escherichia 7.0 ['GCF_013340825.1-NZ_AP023209.1', 'GCF_019970975.1-NZ_CP081709.1', 'GCF_019971015.1-NZ_CP081716.1', 'GCF_022558905.1-NZ_CP060942.1', 'GCF_036494955.1-NZ_AP027498.1', 'GCF_037555585.1-NZ_CP148397.1', 'GCF_039604535.2-NZ_CP166437.1', 'GCF_045344505.1-NZ_CP148748.1', 'GCF_051136185.1-NZ_CP194968.1', 'GCF_964200005.1-NZ_OZ121080.1']
GCF_013340825.1-NZ_AP023209.1 68548 3.0
GCF_019970975.1-NZ_CP081709.1 7467 1.0
GCF_019971015.1-NZ_CP081716.1 37667 3.0
GCF_022558905.1-NZ_CP060942.1 21115 3.0
GCF_036494955.1-NZ_AP027498.1 10618 4.0
GCF_037555585.1-NZ_CP148397.1 1331 7.0
GCF_039604535.2-NZ_CP166437.1 22241 3.0
GCF_045344505.1-NZ_CP148748.1 14065 2.0
GCF_051136185.1-NZ_CP194968.1 1072 7.0
GCF_964200005.1-NZ_OZ121080.1 2586 6.0
Klebsiella 5.0 ['GCF_001663455.1-NZ_CP015133.1', 'GCF_040550265.1-NZ_CP159666.1', 'GCF_044911575.1-NZ_CP110186.1', 'GCF_019286855.1-NZ_CP079163.1', 'GCF_019317225.1-NZ_CP079682.1', 'GCF_030388445.1-NZ_CP122388.1', 'GCF_904864475.1-NZ_LR890473.1', 'GCF_96419

In [53]:
# pick_up_example

genus_name = 'Escherichia'

try:
    layer_link = pd.read_csv(f'{base_folder}/figure_data/circle_layer_network_output/{genus_name}/all_dot_table.csv')
    NMS_link = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/{genus_name}_NMS_replicon_link.csv')
    replicon_data = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    replicon_data['source'] = replicon_data['accession']
    replicon_data['source_size'] = replicon_data['size']
    replicon_data['target'] = replicon_data['accession']
    replicon_data['target_size'] = replicon_data['size']
    size_dict = dict(zip(replicon_data['accession'], replicon_data['size']))
except:
    pass
layer_link['layer_floor'] = layer_link['layer'].apply(np.floor)
unique_layers = layer_link['layer_floor'].unique().tolist()
unique_layers_sorted = sorted(unique_layers)

zero_layer = layer_link[layer_link['layer'] == 0.7]['acc'].to_list()
zero_link = temp_layer_link(zero_layer, layer_link[layer_link['layer_floor'] == 1]['acc'].to_list())
zero_acc = zero_link['source'].unique().tolist()
print(genus_name, max(unique_layers_sorted), zero_acc)
acc = 'GCF_037555585.1-NZ_CP148397.1'
for layer in unique_layers_sorted[1:]:
    if layer == 1:
        previous_layer = [acc]
    else:
        previous_layer = next_layer
    next_layer = layer_link[layer_link['layer_floor'] == layer]['acc'].to_list()
    temp_link = temp_layer_link(previous_layer, next_layer)
    next_layer = temp_link['target'].unique().tolist()
    if len(temp_link) == 0:
        print(acc, size_dict[acc], layer-1)
        break
    elif layer == max(unique_layers_sorted):
        print(acc, size_dict[acc], layer)
    print(temp_link[temp_link['target']=='GCF_036884795.1-NZ_CP145713.1'])
    #print(temp_link)

Escherichia 7.0 ['GCF_013340825.1-NZ_AP023209.1', 'GCF_019971015.1-NZ_CP081716.1', 'GCF_022558905.1-NZ_CP060942.1', 'GCF_036494955.1-NZ_AP027498.1', 'GCF_037555585.1-NZ_CP148397.1', 'GCF_039604535.2-NZ_CP166437.1', 'GCF_045344505.1-NZ_CP148748.1', 'GCF_051136185.1-NZ_CP194968.1', 'GCF_964200005.1-NZ_OZ121080.1']
                             source                         target  coverage  \
1097  GCF_037555585.1-NZ_CP148397.1  GCF_036884795.1-NZ_CP145713.1       1.0   

      source_size  target_size  size_ratio  
1097         1331         5047    0.263721  
Empty DataFrame
Columns: [source, target, coverage, source_size, target_size, size_ratio]
Index: []
Empty DataFrame
Columns: [source, target, coverage, source_size, target_size, size_ratio]
Index: []
Empty DataFrame
Columns: [source, target, coverage, source_size, target_size, size_ratio]
Index: []
Empty DataFrame
Columns: [source, target, coverage, source_size, target_size, size_ratio]
Index: []
Empty DataFrame
Columns: [source, t